# 04.1 – Debug build (Bronze → Silver/Gold)

Notebook miroir du CLI `python -m src.datasets.build --build-from-bronze`, avec quelques impressions supplémentaires pour vérifier rapidement les shapes, les colonnes clés et la présence de NaN. Modifie simplement les variables ci-dessous pour pointer vers des fichiers précis ou limiter le build.

In [ ]:
from pathlib import Path

import pandas as pd

from src.datasets import (
    build_match_dataset_from_bronze,
    build_silver_dataset,
    build_gold_dataset,
    SilverBuildConfig,
    GoldBuildConfig,
)

# ----------------------------
# Config
# ----------------------------
BUILD_FROM_BRONZE = True  # False => charger un fichier match-level existant
MATCHES_PATH = Path("data/01_bronze/matches/YOUR_FILE.parquet")
TARGET = "IS_WIN"
PERSIST_ARTIFACTS = True  # False pour garder tout en mémoire (pas de parquet écrit)

BRONZE_GAMES_PATH = None  # Optionnel, laisser None pour prendre le CSV le plus récent
BRONZE_BOXSCORES_PATH = None  # Optionnel

# ----------------------------
# Bronze → Match-level
# ----------------------------
if BUILD_FROM_BRONZE:
    bronze_result = build_match_dataset_from_bronze(
        games_path=BRONZE_GAMES_PATH,
        boxscores_path=BRONZE_BOXSCORES_PATH,
        persist_artifact=PERSIST_ARTIFACTS,
    )
    matches_df = bronze_result.dataset
    print("[MATCHES] rows=", bronze_result.metadata.get("rows"), "cols=", bronze_result.metadata.get("columns"))
    print("[MATCHES] artifact=", bronze_result.path)
else:
    if MATCHES_PATH.suffix.lower() in {".parquet", ".pq"}:
        matches_df = pd.read_parquet(MATCHES_PATH)
    else:
        matches_df = pd.read_csv(MATCHES_PATH)
    print("[MATCHES] loaded", MATCHES_PATH)
    print(matches_df.shape)



/home/ju/Documents/Dev/NBA_Predictor/src/datasets/bronze.py:78: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)
/home/ju/Documents/Dev/NBA_Predictor/src/feature_aggregation.py:92: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tops = roll.groupby(['GAME_ID', 'TEAM_ID']).apply(pick_top).reset_index()


------------------ Nombre de lignes sans cotes (home/away): 64494 ------------------
[MATCHES] rows= 64494 cols= 230
[MATCHES] artifact= data/01_bronze/matches/bronze_matches_20251108T184555Z.parquet
[MATCHES] sample columns: ['GAME_ID', 'TEAM_ID', 'GAME_DATE', 'SEASON', 'MATCHUP', 'TEAM_NAME', 'TEAM_ABBREVIATION', 'IS_HOME', 'IS_WIN', 'POINTS_FOR', 'OPP_TEAM_ID', 'OPPONENT_NAME', 'OPP_TEAM_ABBREVIATION', 'OPP_PTS', 'POINTS_AGAINST']
[MATCHES] NaN summary:
 IS_HOME           0
IS_WIN            0
POINTS_FOR        0
POINTS_AGAINST    0
dtype: int64


In [3]:

print("[MATCHES] sample columns:", matches_df.columns[:15].tolist())
print("[MATCHES] NaN summary:\n", matches_df[['IS_HOME', 'IS_WIN', 'POINTS_FOR', 'POINTS_AGAINST','MATCHUP']].isna().sum())

[MATCHES] sample columns: ['GAME_ID', 'TEAM_ID', 'GAME_DATE', 'SEASON', 'MATCHUP', 'TEAM_NAME', 'TEAM_ABBREVIATION', 'IS_HOME', 'IS_WIN', 'POINTS_FOR', 'OPP_TEAM_ID', 'OPPONENT_NAME', 'OPP_TEAM_ABBREVIATION', 'OPP_PTS', 'POINTS_AGAINST']
[MATCHES] NaN summary:
 IS_HOME           0
IS_WIN            0
POINTS_FOR        0
POINTS_AGAINST    0
MATCHUP           0
dtype: int64


In [ ]:
""" 
# ----------------------------
# Silver build
# ----------------------------
silver_cfg = SilverBuildConfig()
silver_cfg.persist_artifact = PERSIST_ARTIFACTS
silver_result = build_silver_dataset(matches_df, config=silver_cfg)
silver_df = silver_result.dataset
print("[SILVER] rows=", silver_result.metadata.get("rows"), "cols=", silver_result.metadata.get("columns"))
print("[SILVER] artifact=", silver_result.path)
print("[SILVER] sample columns:", silver_df.columns[:15].tolist())

# ----------------------------
# Gold build
# ----------------------------
gold_cfg = GoldBuildConfig(target=TARGET)
gold_cfg.persist_artifact = PERSIST_ARTIFACTS
gold_result = build_gold_dataset(silver_df, config=gold_cfg)
gold_df = gold_result.dataset
print("[GOLD] rows=", gold_result.metadata.get("rows"), "cols=", gold_result.metadata.get("columns"))
print("[GOLD] artifact=", gold_result.path)
print("[GOLD] NaN target check:", gold_df[TARGET].isna().sum() if TARGET in gold_df.columns else "target dropped")

# Aperçu rapide
display(matches_df.head(2))
display(silver_df.head(2))
display(gold_df.head(2)) """